In [1]:
from pyspark.sql import functions as F

# ============================================================
# NB_Gold_Data_Quality_Summary
# Consolidate Gold data-quality metrics
# ============================================================

# ------------------------------------------------------------
# 1. Load Gold tables
# ------------------------------------------------------------

orders = spark.table("fact_orders")
sales = spark.table("fact_sales")
payments = spark.table("fact_payments")

reconciliation = spark.table("gold_order_reconciliation")

dim_customer = spark.table("dim_customer")
dim_product = spark.table("dim_product")
dim_date = spark.table("dim_date")

# ------------------------------------------------------------
# 2. Calculate quality metrics
# ------------------------------------------------------------

metrics = [
    ("duplicate_customer_keys",
     dim_customer.groupBy("customer_id")
     .count()
     .filter(F.col("count") > 1)
     .count()),

    ("duplicate_product_keys",
     dim_product.groupBy("product_id")
     .count()
     .filter(F.col("count") > 1)
     .count()),

    ("duplicate_order_keys",
     orders.groupBy("order_id")
     .count()
     .filter(F.col("count") > 1)
     .count()),

    ("duplicate_sales_keys",
     sales.groupBy("order_item_id")
     .count()
     .filter(F.col("count") > 1)
     .count()),

    ("duplicate_payment_keys",
     payments.groupBy("payment_id")
     .count()
     .filter(F.col("count") > 1)
     .count()),

    ("orders_null_customer_key",
     orders.filter(F.col("customer_key").isNull()).count()),

    ("orders_null_date_key",
     orders.filter(F.col("date_key").isNull()).count()),

    ("sales_null_customer_key",
     sales.filter(F.col("customer_key").isNull()).count()),

    ("sales_null_product_key",
     sales.filter(F.col("product_key").isNull()).count()),

    ("sales_null_date_key",
     sales.filter(F.col("date_key").isNull()).count()),

    ("payments_null_customer_key",
     payments.filter(F.col("customer_key").isNull()).count()),

    ("payments_null_date_key",
     payments.filter(F.col("date_key").isNull()).count()),

    ("order_reconciliation_matches",
     reconciliation.filter(
         F.col("reconciliation_status") == "MATCH"
     ).count()),

    ("order_reconciliation_mismatches",
     reconciliation.filter(
         F.col("reconciliation_status") == "MISMATCH"
     ).count()),

    ("orders_without_sales_items",
     reconciliation.filter(
         F.col("reconciliation_status") == "NO_SALES_ITEMS"
     ).count())
]

# ------------------------------------------------------------
# 3. Convert metrics into a DataFrame
# ------------------------------------------------------------

metrics_df = spark.createDataFrame(
    metrics,
    ["metric_name", "metric_value"]
)

# ------------------------------------------------------------
# 4. Add status
# ------------------------------------------------------------

metrics_df = metrics_df.withColumn(
    "metric_status",
    F.when(
        F.col("metric_name").isin(
            "order_reconciliation_mismatches",
            "orders_without_sales_items"
        ),
        F.when(
            F.col("metric_value") > 0,
            F.lit("WARNING")
        ).otherwise(
            F.lit("PASS")
        )
    )
    .otherwise(
        F.when(
            F.col("metric_value") == 0,
            F.lit("PASS")
        ).otherwise(
            F.lit("WARNING")
        )
    )
)

# ------------------------------------------------------------
# 5. Add processing date
# ------------------------------------------------------------

metrics_df = metrics_df.withColumn(
    "run_date",
    F.current_timestamp()
)

# ------------------------------------------------------------
# 6. Write Gold table
# ------------------------------------------------------------

metrics_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_data_quality_summary")

# ------------------------------------------------------------
# 7. Display result
# ------------------------------------------------------------

metrics_df.orderBy(
    "metric_name"
).show(
    100,
    truncate=False
)

print("Table created: gold_data_quality_summary")


StatementMeta(, 4f4289a2-6397-4d19-b88e-b4f9ec8fd217, 3, Finished, Available, Finished, False)

+-------------------------------+------------+-------------+--------------------------+
|metric_name                    |metric_value|metric_status|run_date                  |
+-------------------------------+------------+-------------+--------------------------+
|duplicate_customer_keys        |0           |PASS         |2026-09-29 23:51:53.156774|
|duplicate_order_keys           |0           |PASS         |2026-09-29 23:51:53.156774|
|duplicate_payment_keys         |0           |PASS         |2026-09-29 23:51:53.156774|
|duplicate_product_keys         |0           |PASS         |2026-09-29 23:51:53.156774|
|duplicate_sales_keys           |0           |PASS         |2026-09-29 23:51:53.156774|
|order_reconciliation_matches   |1           |WARNING      |2026-09-29 23:51:53.156774|
|order_reconciliation_mismatches|45441       |WARNING      |2026-09-29 23:51:53.156774|
|orders_null_customer_key       |0           |PASS         |2026-09-29 23:51:53.156774|
|orders_null_date_key           